In [29]:
import json
import logging
import pandas as pd
import numpy as np
import pickle
import re
import time
import os
import uuid
import getpass
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any, Tuple
import pubchempy as pcp
from tqdm.notebook import tqdm
import sys
import duckdb

In [30]:
# Creator information
creator_name = getpass.getuser()
timestamp = datetime.now().isoformat()

# Required: Input file with experimental data
input_file_path = "/Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/atlas_tables/HILICZ/HILICZ_QC_POS.tsv"

# Atlas Configuration
create_atlas = True  # If True, creates a named atlas from input compounds
atlas_name = "HILIC Positive QC Default"  # User-defined atlas name
atlas_description = f"Default QC compounds for HILIC Positive as of {timestamp}"  # Optional description

# Required: Output directory for database and cache files
output_dir = Path("/Users/BKieft/Metabolomics/metatlas2/data/databases/compounds")

# Optional: Cache settings
use_cache = True  # Use cached PubChem data if available
cache_filename = "pubchem_global_cache.pkl"  # Single global cache file
force_pubchem_update = False  # If True, forces PubChem query even if data exists in cache

# Optional: Duplicate compound handling
duplicate_compounds = False  # If True, allows adding compounds that already exist in database

## Load and Validate Input Data

In [31]:
# Load and validate input data
print(f"Loading input data from: {input_file_path}")
delimiter = '\t' if input_file_path.endswith(('.tsv', '.tab', '.txt')) else ','
compounds_df = pd.read_csv(input_file_path, sep=delimiter)
print(f"Loaded {len(compounds_df)} rows from input table")

# Check for required columns (only core compound identification needed)
required_columns = ['inchi_key', 'label']
missing_columns = [col for col in required_columns if col not in compounds_df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Get unique InChI keys for PubChem lookup
unique_inchi_keys = compounds_df['inchi_key'].dropna().unique()
print(f"\nUnique InChI keys to process: {len(unique_inchi_keys)}")

Loading input data from: /Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/atlas_tables/HILICZ/HILICZ_QC_POS.tsv
Loaded 20 rows from input table

Unique InChI keys to process: 20


## PubChem Data Retrieval Functions

In [32]:
def get_pubchem_data(inchi_key: str) -> Dict[str, Any]:
    """Get comprehensive compound data from PubChem using InChI key."""
    try:

        # Get CID from InChI key
        cid_result = pcp.get_compounds(inchi_key, namespace='inchikey', 
                                        as_dataframe=True, listkey_count=5)

        if cid_result.empty:
            return None
            
        cid_result = cid_result.reset_index()
        cid = cid_result['cid'].to_string(index=False)
        
        # Handle multiple CIDs
        if "\n" in cid:
            cid = (cid.rstrip().split('\n'))[-1]
        
        # Extract SMILES if available due to broken API response
        try:
            cid_result_subset = cid_result[cid_result['cid'] == int(cid)]
            cid_result_subset_dict = cid_result_subset.to_dict()
            if "record" in cid_result_subset_dict:
                if "props" in cid_result_subset_dict["record"][0]:
                    cid_list = [i for i in cid_result_subset_dict["record"][0]["props"] if i["urn"]["label"] == "SMILES"]
                    if cid_list:
                        smiles = str(cid_list[0]['value']['sval'])
        except:
            smiles = ""

        # Get detailed compound information
        compound = pcp.Compound.from_cid(cid)
        
        # Extract all available properties
        compound_data = {
            "pubchem_cid": str(compound.cid) if compound.cid else "",
            "iupac_name": compound.iupac_name or "",
            "synonyms": compound.synonyms or [],
            "inchi": compound.inchi or "",
            "inchi_key": compound.inchikey or inchi_key,
            "smiles": smiles if smiles else "",
            "formula": compound.molecular_formula or "",
            "mono_isotopic_molecular_weight": str(compound.monoisotopic_mass) if compound.monoisotopic_mass else "",
            "molecular_weight": str(compound.molecular_weight) if compound.molecular_weight else "",
            "cas_number": "",
            "pubchem_retrieval_date": timestamp,
            "pubchem_compound_url": f"https://pubchem.ncbi.nlm.nih.gov/compound/{compound.cid}" if compound.cid else ""
        }
        
        # Extract CAS number from synonyms
        if compound.synonyms:
            for synonym in compound.synonyms:
                if not compound_data["cas_number"] and '-' in synonym and len(synonym.split('-')) == 3:
                    parts = synonym.split('-')
                    if all(part.isdigit() for part in parts):
                        compound_data["cas_number"] = synonym
                        break
        
        return compound_data
        
    except Exception as e:
        print(f"Error retrieving PubChem data for {inchi_key}: {e}")
        return None

def filter_synonym_list(synonyms: List[str]) -> str:
    """Filter synonym list to find the best name."""
    if not synonyms or synonyms == ["Undefined"]:
        return "Undefined"
    
    if isinstance(synonyms, str):
        synonyms = [synonyms]
    
    problematic_prefixes = (
        "nan", "Untitled", "Oprea", "Opera", "AKO", "CHEMBL", "SR-", "SCHEM", 
        "EU-", "MLS", "NSC", "ChemDiv", "ST0", "TimTec", "HMS", "BIM", "CB", 
        "CCG-", "Cambridge", "SMR", "AB0", "BRD-", "NCG", "BDBM", "CBKinase", 
        "BAS ", "ZINC", "GNF", "SQX", "CDS", "STK", "NCI", "TNP", "Boc-Tyr-OH", 
        "PD", "UNM", "BSP", "CCRIS", "MFCD", "IDI", "ST5", "AC1", "WAY-", "KUC", 
        "DTXSID", "MixCom", "CK-", "ASN ", "MMV", "SKI-", "VU", "SMSF", "Bio2", 
        "REGID", "SDCC", "BCBc", "SMP", "TCMDC", "cid_", "BCP", "AST ", "SY0", 
        "AM-", "IFLab", "Cream"
    )

    # Remove problematic prefixes
    filtered_synonyms = [x for x in synonyms if not x.startswith(problematic_prefixes)]
    filtered_synonyms = [x for x in filtered_synonyms if not "cream" in x.lower()]
    
    # Remove entries that are mostly digits or codes
    filtered_synonyms = [x for x in filtered_synonyms 
                        if not x.replace("-", "").replace(re.compile('^A-Z').pattern, "").isdigit()]
    
    if not filtered_synonyms:
        return "Undefined"
    
    # Return the shortest remaining synonym
    return min(filtered_synonyms, key=len)

def load_or_create_pubchem_cache() -> Dict[str, Dict]:
    """Load existing global PubChem cache or create new one"""
    cache_file = output_dir / "pubchem_cache" / cache_filename
    
    if use_cache and cache_file.exists():
        try:
            with open(cache_file, 'rb') as f:
                cache = pickle.load(f)
            print(f"Loaded global PubChem cache with {len(cache)} entries from {cache_file}")
            return cache
        except Exception as e:
            print(f"Error loading cache: {e}. Creating new cache.")
    
    print("Creating new global PubChem cache")
    return {}

def save_pubchem_cache(cache: Dict[str, Dict]) -> None:
    """Save global PubChem cache to file"""
    cache_file = output_dir / "pubchem_cache" / cache_filename
    cache_file.parent.mkdir(parents=True, exist_ok=True)
    
    try:
        with open(cache_file, 'wb') as f:
            pickle.dump(cache, f)
        print(f"Saved global PubChem cache with {len(cache)} entries to {cache_file}")
    except Exception as e:
        print(f"Error saving cache: {e}")

def create_metatlas_database(input_df: pd.DataFrame, pubchem_cache: Dict, db_path: Path):
    """Create a single DuckDB with all metabolomics data"""
    print(f"Creating database: {db_path}")
    
    conn = duckdb.connect(str(db_path))
    
    # Create tables
    conn.execute("""
        CREATE TABLE IF NOT EXISTS compounds (
            compound_uid VARCHAR PRIMARY KEY,
            name VARCHAR NOT NULL,
            inchi_key VARCHAR(27) NOT NULL UNIQUE,
            inchi TEXT,
            smiles TEXT,
            formula VARCHAR,
            compound_classes TEXT,
            compound_pathways TEXT,
            compound_tags TEXT,
            molecular_weight DOUBLE,
            mono_isotopic_molecular_weight DOUBLE,
            iupac_name TEXT,
            pubchem_cid VARCHAR,
            cas_number VARCHAR,
            synonyms TEXT,
            created_by VARCHAR,
            creation_time TIMESTAMP,
            last_modified TIMESTAMP
        )
    """)
    
    # Create atlas tables
    conn.execute("""
        CREATE TABLE IF NOT EXISTS atlases (
            atlas_uid VARCHAR PRIMARY KEY,
            atlas_name VARCHAR NOT NULL,
            atlas_description TEXT,
            chromatography VARCHAR,
            polarity VARCHAR,
            created_by VARCHAR,
            creation_time TIMESTAMP,
            last_modified TIMESTAMP,
            UNIQUE(atlas_name, chromatography, polarity)
        )
    """)
    
    conn.execute("""
        CREATE TABLE IF NOT EXISTS mz_rt_references (
            mz_rt_reference_uid VARCHAR PRIMARY KEY,
            compound_uid VARCHAR NOT NULL,
            rt_peak DOUBLE NOT NULL,
            rt_min DOUBLE NOT NULL,
            rt_max DOUBLE NOT NULL,
            mz DOUBLE NOT NULL,
            mz_tolerance DOUBLE NOT NULL,
            adduct VARCHAR NOT NULL,
            chromatography VARCHAR NOT NULL,
            polarity VARCHAR NOT NULL,
            confidence VARCHAR,
            source VARCHAR,
            created_by VARCHAR,
            creation_time TIMESTAMP,
            FOREIGN KEY (compound_uid) REFERENCES compounds(compound_uid)
        )
    """)

    conn.execute("""
        CREATE TABLE IF NOT EXISTS atlas_compound_associations (
            association_uid VARCHAR PRIMARY KEY,
            atlas_uid VARCHAR NOT NULL,
            compound_uid VARCHAR NOT NULL,
            mz_rt_reference_uid VARCHAR,
            association_order INTEGER,
            created_by VARCHAR,
            creation_time TIMESTAMP,
            FOREIGN KEY (atlas_uid) REFERENCES atlases(atlas_uid),
            FOREIGN KEY (compound_uid) REFERENCES compounds(compound_uid),
            FOREIGN KEY (mz_rt_reference_uid) REFERENCES mz_rt_references(mz_rt_reference_uid),
            UNIQUE(atlas_uid, compound_uid, mz_rt_reference_uid)
        )
    """)
    
    # Create indexes
    conn.execute("CREATE INDEX IF NOT EXISTS idx_compounds_inchi_key ON compounds(inchi_key)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_mzrt_compound ON mz_rt_references(compound_uid)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_mzrt_method ON mz_rt_references(chromatography, polarity)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_atlases_name ON atlases(atlas_name)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_atlas_associations_atlas ON atlas_compound_associations(atlas_uid)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_atlas_associations_compound ON atlas_compound_associations(compound_uid)")
    
    # Get existing compounds
    existing_inchi_keys = set()
    if not duplicate_compounds:
        # Get existing compounds
        existing_result = conn.execute("SELECT inchi_key FROM compounds").fetchall()
        existing_inchi_keys = {row[0] for row in existing_result}
        
        print(f"Found {len(existing_inchi_keys)} existing compounds in database")
    
    # Process compounds and references
    compounds_created = 0
    compounds_skipped = 0
    references_created = 0
    references_skipped = 0
    
    # Create atlas if requested - NEW
    atlas_uid = None
    if create_atlas:
        atlas_uid = create_or_get_atlas(conn, atlas_name, atlas_description, 
                                       input_df.get('chromatography', 'Unknown').iloc[0] if not input_df.empty else 'Unknown',
                                       input_df.get('polarity', 'Unknown').iloc[0] if not input_df.empty else 'Unknown')
    
    atlas_associations_created = 0
    
    for idx, row in tqdm(input_df.iterrows(), total=len(input_df), desc="Processing compounds"):
        inchi_key = row.get('inchi_key')
        if pd.isna(inchi_key):
            continue
        
        # Get method information from input data
        chromatography = str(row.get('chromatography', 'Unknown'))
        polarity = str(row.get('polarity', 'Unknown'))
        
        compound_uid = None
        
        # Check if compound exists
        if not duplicate_compounds and inchi_key in existing_inchi_keys:
            compounds_skipped += 1
            
            # Get existing compound_uid
            existing_compound_result = conn.execute(
                "SELECT compound_uid FROM compounds WHERE inchi_key = ?", 
                [inchi_key]
            ).fetchone()
            
            if existing_compound_result:
                compound_uid = existing_compound_result[0]
        else:
            # Create new compound
            pubchem_data = pubchem_cache.get(inchi_key, {})
            tag_data = input_df.loc[idx, ['compound_classes', 'compound_pathways', 'compound_tags']].to_dict()
            compound_uid = f"comp-{uuid.uuid4().hex[:16]}"
            
            # Convert molecular weights
            mol_weight = None
            mono_weight = None
            if pubchem_data.get('molecular_weight'):
                try:
                    mol_weight = float(pubchem_data['molecular_weight'])
                except (ValueError, TypeError):
                    pass
            if pubchem_data.get('mono_isotopic_molecular_weight'):
                try:
                    mono_weight = float(pubchem_data['mono_isotopic_molecular_weight'])
                except (ValueError, TypeError):
                    pass
            
            conn.execute("""
                INSERT INTO compounds VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                compound_uid,
                str(row.get('label', "Undefined")),
                inchi_key,
                pubchem_data.get('inchi', ''),
                pubchem_data.get('smiles', ''),
                pubchem_data.get('formula', ''),
                tag_data.get('compound_classes', ''),
                tag_data.get('compound_pathways', ''),
                tag_data.get('compound_tags', ''),
                mol_weight,
                mono_weight,
                pubchem_data.get('iupac_name', ''),
                pubchem_data.get('pubchem_cid', ''),
                pubchem_data.get('cas_number', ''),
                "; ".join(pubchem_data.get('synonyms', [])) if pubchem_data.get('synonyms') else '',
                creator_name,
                timestamp,
                timestamp
            ))
            compounds_created += 1
            
            # Add to existing sets for future duplicate checking in this session
            if not duplicate_compounds:
                existing_inchi_keys.add(inchi_key)
        
        # Create RT/MZ reference if data available and compound_uid exists
        mz_rt_ref_uid = None
        if compound_uid and pd.notna(row.get('rt_peak')) and pd.notna(row.get('mz')):
            rt_peak = float(row['rt_peak'])
            mz = float(row['mz'])
            adduct = str(row.get('adduct', '[M+H]+'))
            
            # Check if similar RT/MZ reference already exists for this compound and method
            rt_tolerance = 0.1  # Allow small RT difference
            mz_tolerance_val = float(row.get('mz_tolerance', 20.0))
            mz_tolerance_ppm = mz_tolerance_val if mz_tolerance_val > 1 else mz_tolerance_val * 1e6
            mz_delta = (mz * mz_tolerance_ppm) / 1e6
            
            existing_ref_result = conn.execute("""
                SELECT mz_rt_reference_uid FROM mz_rt_references 
                WHERE compound_uid = ? 
                AND chromatography = ? 
                AND polarity = ?
                AND ABS(rt_peak - ?) <= ?
                AND ABS(mz - ?) <= ?
                AND adduct = ?
            """, [
                compound_uid, 
                chromatography, 
                polarity,
                rt_peak, rt_tolerance,
                mz, mz_delta,
                adduct
            ]).fetchone()
            
            if existing_ref_result:
                # RT/MZ reference already exists with similar values
                references_skipped += 1
                mz_rt_ref_uid = existing_ref_result[0]
            else:
                # Create new RT/MZ reference
                mz_rt_ref_uid = f"mzrt-{uuid.uuid4().hex[:16]}"
                
                rt_min = float(row.get('rt_min', rt_peak - 0.5))
                rt_max = float(row.get('rt_max', rt_peak + 0.5))
                
                conn.execute("""
                    INSERT INTO mz_rt_references VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """, (
                    mz_rt_ref_uid,
                    compound_uid,
                    rt_peak,
                    rt_min,
                    rt_max,
                    mz,
                    float(row.get('mz_tolerance', 20.0)),
                    adduct,
                    chromatography,
                    polarity,
                    row.get('confidence', ''),
                    os.path.basename(input_file_path),
                    creator_name,
                    timestamp
                ))
                references_created += 1
        
        # Associate compound and mz_rt_reference with atlas - NEW
        if create_atlas and atlas_uid and compound_uid:
            association_uid = f"assoc-{uuid.uuid4().hex[:16]}"
            try:
                conn.execute("""
                    INSERT INTO atlas_compound_associations VALUES (?, ?, ?, ?, ?, ?, ?)
                """, (
                    association_uid,
                    atlas_uid,
                    compound_uid,
                    mz_rt_ref_uid,  # Can be None if no RT/MZ reference
                    idx + 1,  # association_order based on input file order
                    creator_name,
                    timestamp
                ))
                atlas_associations_created += 1
            except Exception as e:
                # Handle duplicate associations gracefully
                if "UNIQUE constraint failed" not in str(e):
                    print(f"Warning: Could not create atlas association for {row.get('label', 'Unknown')}: {e}")
    
    # Final counts
    compounds_count = conn.execute("SELECT COUNT(*) FROM compounds").fetchone()[0]
    references_count = conn.execute("SELECT COUNT(*) FROM mz_rt_references").fetchone()[0]
    
    print(f"\n✓ Database created successfully!")
    print(f"   Total compounds in database: {compounds_count}")
    print(f"   New compounds created: {compounds_created}")
    if compounds_skipped > 0:
        print(f"   Compounds skipped (already existed): {compounds_skipped}")
    print(f"   Total RT/MZ references in database: {references_count}")
    print(f"   New RT/MZ references created: {references_created}")
    if references_skipped > 0:
        print(f"   RT/MZ references skipped (duplicates): {references_skipped}")
    
    if create_atlas and atlas_uid:
        print(f"   Atlas '{atlas_name}' created with UID: {atlas_uid}")
        print(f"   Atlas associations created: {atlas_associations_created}")
    
    conn.close()
    return compounds_created, references_created, compounds_skipped, references_skipped, atlas_uid

def create_or_get_atlas(conn, atlas_name: str, atlas_description: str, 
                       chromatography: str, polarity: str) -> str:
    """Create a new atlas or get existing atlas UID."""
    
    # Check if atlas already exists
    existing_atlas = conn.execute("""
        SELECT atlas_uid FROM atlases 
        WHERE atlas_name = ? AND chromatography = ? AND polarity = ?
    """, [atlas_name, chromatography, polarity]).fetchone()
    
    if existing_atlas:
        atlas_uid = existing_atlas[0]
        print(f"Using existing atlas: {atlas_name} (UID: {atlas_uid})")
        
        # Update last_modified timestamp
        conn.execute("""
            UPDATE atlases SET last_modified = ? WHERE atlas_uid = ?
        """, [timestamp, atlas_uid])
        
        return atlas_uid
    
    # Create new atlas
    atlas_uid = f"atlas-{uuid.uuid4().hex[:16]}"
    
    conn.execute("""
        INSERT INTO atlases VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        atlas_uid,
        atlas_name,
        atlas_description,
        chromatography,
        polarity,
        creator_name,
        timestamp,
        timestamp
    ))
    
    print(f"Created new atlas: {atlas_name} (UID: {atlas_uid})")
    return atlas_uid

def get_atlas_compounds(db_path: str, atlas_uid: str = None, atlas_name: str = None, 
                       chromatography: str = None, polarity: str = None) -> pd.DataFrame:
    """
    Retrieve compounds associated with a specific atlas.
    
    Args:
        db_path: Path to the database
        atlas_uid: Specific atlas UID (highest priority)
        atlas_name: Atlas name (used with atlas_type, chromatography, polarity)
        chromatography: Chromatography method
        polarity: MS polarity
    
    Returns:
        DataFrame with compound and RT/MZ reference information
    """
    conn = duckdb.connect(str(db_path))
    
    if atlas_uid:
        # Get compounds by specific atlas UID
        query = """
        SELECT 
            a.atlas_uid,
            a.atlas_name,
            c.compound_uid,
            c.name as compound_name,
            c.inchi_key,
            mzrt.mz_rt_reference_uid,
            mzrt.rt_peak,
            mzrt.rt_min,
            mzrt.rt_max,
            mzrt.mz,
            mzrt.mz_tolerance,
            mzrt.adduct,
            mzrt.chromatography,
            mzrt.polarity,
            aca.association_order
        FROM atlases a
        JOIN atlas_compound_associations aca ON a.atlas_uid = aca.atlas_uid
        JOIN compounds c ON aca.compound_uid = c.compound_uid
        LEFT JOIN mz_rt_references mzrt ON aca.mz_rt_reference_uid = mzrt.mz_rt_reference_uid
        WHERE a.atlas_uid = ?
        ORDER BY aca.association_order
        """
        df = conn.execute(query, [atlas_uid]).df()
        
    else:
        # Get compounds by atlas name and filters
        conditions = ["1=1"]
        params = []
        
        if atlas_name:
            conditions.append("a.atlas_name = ?")
            params.append(atlas_name)
        if chromatography:
            conditions.append("a.chromatography = ?")
            params.append(chromatography)
        if polarity:
            conditions.append("a.polarity = ?")
            params.append(polarity)
            
        query = f"""
        SELECT 
            a.atlas_uid,
            a.atlas_name,
            c.compound_uid,
            c.name as compound_name,
            c.inchi_key,
            mzrt.mz_rt_reference_uid,
            mzrt.rt_peak,
            mzrt.rt_min,
            mzrt.rt_max,
            mzrt.mz,
            mzrt.mz_tolerance,
            mzrt.adduct,
            mzrt.chromatography,
            mzrt.polarity,
            aca.association_order
        FROM atlases a
        JOIN atlas_compound_associations aca ON a.atlas_uid = aca.atlas_uid
        JOIN compounds c ON aca.compound_uid = c.compound_uid
        LEFT JOIN mz_rt_references mzrt ON aca.mz_rt_reference_uid = mzrt.mz_rt_reference_uid
        WHERE {' AND '.join(conditions)}
        ORDER BY a.atlas_name, aca.association_order
        """
        df = conn.execute(query, params).df()
    
    conn.close()
    
    if len(df) == 0:
        print(f"No compounds found for the specified atlas criteria")
    else:
        print(f"Retrieved {len(df)} compounds from atlas")
    
    return df

def list_available_atlases(db_path: str, chromatography: str = None, polarity: str = None) -> pd.DataFrame:
    """
    List all available atlases with optional filtering.
    
    Returns:
        DataFrame with atlas information and compound counts
    """
    conn = duckdb.connect(str(db_path))
    
    conditions = ["1=1"]
    params = []
    
    if chromatography:
        conditions.append("a.chromatography = ?")
        params.append(chromatography)
    if polarity:
        conditions.append("a.polarity = ?")
        params.append(polarity)
    
    query = f"""
    SELECT 
        a.atlas_uid,
        a.atlas_name,
        a.atlas_description,
        a.chromatography,
        a.polarity,
        a.created_by,
        a.creation_time,
        a.last_modified,
        COUNT(aca.compound_uid) as compound_count
    FROM atlases a
    LEFT JOIN atlas_compound_associations aca ON a.atlas_uid = aca.atlas_uid
    WHERE {' AND '.join(conditions)}
    GROUP BY a.atlas_uid, a.atlas_name, a.atlas_description, 
             a.chromatography, a.polarity, a.created_by, a.creation_time, a.last_modified
    ORDER BY a.atlas_name
    """
    
    df = conn.execute(query, params).df()
    conn.close()
    
    return df

## Retrieve PubChem Data for All Compounds

In [33]:
# Load existing global cache
pubchem_cache = load_or_create_pubchem_cache()

# Determine which compounds need PubChem lookup
if force_pubchem_update:
    # Force update: query all compounds regardless of cache status
    compounds_to_fetch = list(unique_inchi_keys)
    compounds_in_cache = []
    print(f"Force update enabled - will query all {len(compounds_to_fetch)} compounds")
else:
    # Normal mode: only query compounds not in cache
    compounds_to_fetch = [key for key in unique_inchi_keys if key not in pubchem_cache]
    compounds_in_cache = [key for key in unique_inchi_keys if key in pubchem_cache]
    print(f"Compounds already in cache: {len(compounds_in_cache)}")
    print(f"Compounds needing PubChem lookup: {len(compounds_to_fetch)}")

if compounds_to_fetch:
    print(f"\nFetching PubChem data for {len(compounds_to_fetch)} compounds...")
    if force_pubchem_update:
        print("Force update mode: updating existing cache entries")
    print("This may take several minutes depending on the number of compounds.")
    
    # Track how many were actually updated
    new_entries = 0
    updated_entries = 0
    
    # Fetch data for compounds
    for inchi_key in tqdm(compounds_to_fetch, desc="Fetching PubChem data"):
        
        was_in_cache = inchi_key in pubchem_cache
        
        # Get PubChem data
        pubchem_data = get_pubchem_data(inchi_key)
        
        if pubchem_data:
            # Add retrieval timestamp to track when data was last updated
            pubchem_data["last_updated"] = timestamp
            pubchem_cache[inchi_key] = pubchem_data
            
            if was_in_cache:
                updated_entries += 1
            else:
                new_entries += 1
        else:
            print(f"No PubChem data found for {inchi_key}")
            # Store empty entry to avoid re-querying (unless force update is used)
            empty_entry = {
                "inchi_key": inchi_key,
                "pubchem_cid": "",
                "iupac_name": "",
                "synonyms": ["Undefined"],
                "error": "not_found_in_pubchem",
                "last_updated": timestamp
            }
            
            # Only store empty entry if not forcing updates or if it's truly new
            if not was_in_cache:
                pubchem_cache[inchi_key] = empty_entry
                new_entries += 1
        
        # Be respectful to PubChem API
        time.sleep(0.25)
    
    # Save updated cache
    save_pubchem_cache(pubchem_cache)
    
    # Report what was done
    if force_pubchem_update:
        print(f"✓ Force update completed: {updated_entries} entries updated, {new_entries} entries added")
    else:
        print(f"✓ Cache update completed: {new_entries} new entries added")
    
else:
    print("All compounds already in cache!")

print(f"\nPubChem data retrieval complete!")
print(f"Total compounds in global cache: {len(pubchem_cache)}")

# Show examples
successful_retrievals = [k for k, v in pubchem_cache.items() 
                        if v.get('pubchem_cid') and v.get('pubchem_cid') != '']
failed_retrievals = [k for k, v in pubchem_cache.items() 
                    if not v.get('pubchem_cid') or v.get('pubchem_cid') == '']

print(f"Successful PubChem retrievals in cache: {len(successful_retrievals)}")
print(f"Failed retrievals in cache: {len(failed_retrievals)}")

# Show cache age info if available
entries_with_dates = [v for v in pubchem_cache.values() if v.get('last_updated')]
if entries_with_dates:
    print(f"Cache entries with timestamps: {len(entries_with_dates)}")

if failed_retrievals:
    print(f"\nSome compounds not found in PubChem: {failed_retrievals[:5]}...")
    print("These will be created with minimal information from the input table.")

Loaded global PubChem cache with 482 entries from /Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/pubchem_cache/pubchem_global_cache.pkl
Compounds already in cache: 20
Compounds needing PubChem lookup: 0
All compounds already in cache!

PubChem data retrieval complete!
Total compounds in global cache: 482
Successful PubChem retrievals in cache: 478
Failed retrievals in cache: 4
Cache entries with timestamps: 482

Some compounds not found in PubChem: ['HEBKCHPVOIAQTA-OWTCIZGHSA-N', 'HEBKCHPVOIAQTA-ZXFHETKHSA-N', 'HEBKCHPVOIAQTA-FCLZTKDGSA-N', 'HEBKCHPVOIAQTA-QSYSQFBZSA-N']...
These will be created with minimal information from the input table.


## Create Database

In [34]:
# Prepare compounds for database creation
print(f"Preparing {len(unique_inchi_keys)} compounds for database creation...")

compounds = {}
for idx, row in compounds_df.iterrows():
    inchi_key = row.get('inchi_key')
    if pd.notna(inchi_key):
        compounds[inchi_key] = row.to_dict()
        
# Create DuckDB database
print(f"\nCreating DuckDB database...")

db_path = output_dir / "metatlas.duckdb"
output_dir.mkdir(exist_ok=True)

# Create the database
compounds_created, references_created, compounds_skipped, references_skipped, atlas_uid = create_metatlas_database(compounds_df, pubchem_cache, db_path)

print(f"\n✓ Database creation complete!")
print(f"   Database file: {db_path}")
print(f"   Created {compounds_created} new compounds")
print(f"   Skipped {compounds_skipped} duplicate compounds")
print(f"   Created {references_created} new RT/MZ references")
print(f"   Skipped {references_skipped} duplicate RT/MZ references")

if create_atlas and atlas_uid:
    print(f"\n✓ Atlas creation complete!")
    print(f"   Atlas Name: {atlas_name}")
    print(f"   Atlas UID: {atlas_uid}")

Preparing 20 compounds for database creation...

Creating DuckDB database...
Creating database: /Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/metatlas.duckdb
Found 482 existing compounds in database
Created new atlas: HILIC Positive QC Default (UID: atlas-b8ae18bcb00549b7)


Processing compounds:   0%|          | 0/20 [00:00<?, ?it/s]


✓ Database created successfully!
   Total compounds in database: 482
   New compounds created: 0
   Compounds skipped (already existed): 20
   Total RT/MZ references in database: 872
   New RT/MZ references created: 0
   RT/MZ references skipped (duplicates): 20
   Atlas 'HILIC Positive QC Default' created with UID: atlas-b8ae18bcb00549b7
   Atlas associations created: 20

✓ Database creation complete!
   Database file: /Users/BKieft/Metabolomics/metatlas2/data/databases/compounds/metatlas.duckdb
   Created 0 new compounds
   Skipped 20 duplicate compounds
   Created 0 new RT/MZ references
   Skipped 20 duplicate RT/MZ references

✓ Atlas creation complete!
   Atlas Name: HILIC Positive QC Default
   Atlas UID: atlas-b8ae18bcb00549b7


In [35]:
# Optional: Quick database validation
print(f"\nDatabase Validation:")

try:
    conn = duckdb.connect(str(db_path))
    
    # Show table counts
    compounds_count = conn.execute("SELECT COUNT(*) FROM compounds").fetchone()[0]
    references_count = conn.execute("SELECT COUNT(*) FROM mz_rt_references").fetchone()[0]
    atlases_count = conn.execute("SELECT COUNT(*) FROM atlases").fetchone()[0]
    atlas_compound_associations_count = conn.execute("SELECT COUNT(*) FROM atlas_compound_associations").fetchone()[0]
    
    # Show method combinations
    method_combinations = conn.execute("""
        SELECT chromatography, polarity, COUNT(*) as reference_count 
        FROM mz_rt_references 
        GROUP BY chromatography, polarity
    """).fetchall()

    # Show atlas information (include atlas_uid)
    atlas_info = conn.execute("""
        SELECT a.atlas_uid, atlas_name, chromatography, polarity, COUNT(aca.compound_uid) as compound_count
        FROM atlases a
        LEFT JOIN atlas_compound_associations aca ON a.atlas_uid = aca.atlas_uid
        GROUP BY a.atlas_uid, atlas_name, chromatography, polarity
        ORDER BY atlas_name
    """).fetchall()

    print(f"   Compounds: {compounds_count}")
    print(f"   RT/MZ References: {references_count}")
    print(f"   Atlases: {atlases_count}")
    print(f"   Atlas-Compound associations: {atlas_compound_associations_count}")
    
    print(f"   Method combinations:")
    for combo in method_combinations:
        print(f"      {combo[0]}/{combo[1]}: {combo[2]} references")
    
    if atlas_info:
        print(f"\n   Available atlases:")
        for atlas_uid, atlas_name, chromatography, polarity, compound_count in atlas_info:
            print(f"      {atlas_name} (UID: {atlas_uid}) ({chromatography}/{polarity}): {compound_count} compounds")
    else:
        print(f"\n   No atlases found")

    conn.close()
    
except Exception as e:
    print(f"Error validating database: {e}")


Database Validation:
   Compounds: 482
   RT/MZ References: 872
   Atlases: 5
   Atlas-Compound associations: 961
   Method combinations:
      HILICZ/negative: 466 references
      HILICZ/positive: 406 references

   Available atlases:
      HILIC Negative EMA Default (UID: atlas-9ea15e8a2be84925) (HILICZ/negative): 418 compounds
      HILIC Negative ISTD Default (UID: atlas-e641984aae9947bc) (HILICZ/negative): 85 compounds
      HILIC Positive EMA Default (UID: atlas-b744d9296ae848a5) (HILICZ/positive): 373 compounds
      HILIC Positive ISTD Default (UID: atlas-62499fad4e104743) (HILICZ/positive): 65 compounds
      HILIC Positive QC Default (UID: atlas-b8ae18bcb00549b7) (HILICZ/positive): 20 compounds
